# 02 · Regresión logística

## Objetivo

En este notebook vamos a construir y entender un modelo de **regresión logística** usando un ejemplo sintético inspirado en ATM.

Supongamos que queremos estimar si una ventana operativa de 15 minutos tendrá una **situación de alta carga de coordinación**.

El target será binario:

- `0`: carga normal
- `1`: alta carga

Usaremos inicialmente una sola variable:

- `flights_15min`: número de vuelos en la ventana

El objetivo es comprender:

- por qué la regresión logística es un modelo de clasificación;
- qué es la función sigmoide;
- qué significa que el modelo produzca una probabilidad;
- qué papel juega el umbral de decisión;
- cómo interpretar coeficientes;
- cómo evaluar un clasificador;
- qué son accuracy, precision, recall y F1;
- cómo leer una matriz de confusión;
- y por qué elegir el umbral depende del problema.


## 1. Importaciones


In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Crear un conjunto de datos ATM sintético

Generaremos ventanas de 15 minutos con distinto número de vuelos.

La probabilidad de alta carga aumentará con el tráfico, pero no de forma determinista.

Esto es importante: para un mismo número de vuelos puede haber ventanas normales y ventanas de alta carga, porque en un sistema real influirían también otros factores.

Para este ejercicio simplificaremos todo eso en ruido probabilístico.


In [ ]:
rng = np.random.default_rng(42)

n_samples = 300
flights_15min = rng.integers(5, 36, size=n_samples)

# La transición importante ocurre aproximadamente alrededor de 20 vuelos.
logit = -8.0 + 0.40 * flights_15min
prob_high_load = 1 / (1 + np.exp(-logit))

high_load = rng.binomial(1, prob_high_load)

df = pd.DataFrame({
    "flights_15min": flights_15min,
    "high_load": high_load
})

df.head()


## 3. Explorar los datos


In [ ]:
df.describe()


In [ ]:
df["high_load"].value_counts().sort_index()


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["flights_15min"], df["high_load"], alpha=0.35)
plt.yticks([0, 1], ["Normal", "Alta carga"])
plt.xlabel("Vuelos por 15 min")
plt.ylabel("Clase")
plt.title("Datos sintéticos ATM")
plt.grid(alpha=0.2)
plt.show()


## 4. ¿Por qué no usar una regresión lineal?

La regresión logística parte de una combinación lineal:

$$
z = b + wx
$$

pero transforma esa salida mediante la función sigmoide:

$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

La sigmoide siempre produce valores entre 0 y 1, por lo que podemos interpretarlos como probabilidades.


In [ ]:
z = np.linspace(-10, 10, 400)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 5))
plt.plot(z, sigmoid)
plt.axhline(0.5, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("z = b + wx")
plt.ylabel("sigmoid(z)")
plt.title("Función sigmoide")
plt.grid(alpha=0.2)
plt.show()


## 5. Separar `X` e `y`


In [ ]:
X = df[["flights_15min"]]
y = df["high_load"]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)


## 6. Train / test split

Usaremos 80 % para entrenamiento y 20 % para test.

`stratify=y` ayuda a mantener una proporción de clases similar en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape[0], "muestras")
print("Test :", X_test.shape[0], "muestras")
print("Proporción positiva train:", round(y_train.mean(), 3))
print("Proporción positiva test :", round(y_test.mean(), 3))


## 7. Entrenar la regresión logística


In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)


In [ ]:
intercept = model.intercept_[0]
coef = model.coef_[0][0]

print(f"Intercepto: {intercept:.4f}")
print(f"Coeficiente: {coef:.4f}")
print(f"Odds ratio por vuelo adicional: {np.exp(coef):.3f}")


El coeficiente no representa directamente un cambio en probabilidad.

La regresión logística modela:

$$
\log\left(\frac{p}{1-p}\right)=b+wx
$$

Por eso, `exp(coeficiente)` se interpreta como un **odds ratio**.


## 8. Obtener probabilidades


In [ ]:
probabilities = model.predict_proba(X_test)
p_high_load = probabilities[:, 1]

results = pd.DataFrame({
    "flights_15min": X_test["flights_15min"].to_numpy(),
    "actual": y_test.to_numpy(),
    "p_high_load": p_high_load
})

results.head(10)


## 9. Convertir probabilidad en clase

Por defecto, `scikit-learn` usa un umbral de 0.5:

- si `P(y=1) >= 0.5` → clase 1
- si `P(y=1) < 0.5` → clase 0


In [ ]:
y_pred = model.predict(X_test)
results["predicted"] = y_pred
results.head(10)


## 10. Visualizar la curva logística


In [ ]:
x_line = np.linspace(df["flights_15min"].min(), df["flights_15min"].max(), 300)
x_line_df = pd.DataFrame({"flights_15min": x_line})
p_line = model.predict_proba(x_line_df)[:, 1]

plt.figure(figsize=(9, 5))
plt.scatter(df["flights_15min"], df["high_load"], alpha=0.22, label="Observaciones")
plt.plot(x_line, p_line, linewidth=2.5, label="P(alta carga)")
plt.axhline(0.5, linestyle="--", label="Umbral 0.5")
plt.xlabel("Vuelos por 15 min")
plt.ylabel("Probabilidad / clase")
plt.title("Regresión logística")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 11. Frontera de decisión


In [ ]:
decision_boundary = -intercept / coef
print(f"Frontera aproximada con umbral 0.5: {decision_boundary:.2f} vuelos")


La frontera aparece donde:

$$
P(y=1)=0.5
$$

y, por tanto:

$$
b+wx=0
$$


## 12. Métricas de clasificación


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1       : {f1:.3f}")


- **Accuracy**: proporción total de aciertos.
- **Precision**: de los casos predichos como positivos, cuántos eran realmente positivos.
- **Recall**: de todos los positivos reales, cuántos detectamos.
- **F1**: combina precision y recall.

La accuracy puede ser engañosa si las clases están muy desbalanceadas.


## 13. Matriz de confusión


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm


In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Alta carga"]
)
disp.plot()
plt.title("Matriz de confusión")
plt.show()


Para la clase positiva `1 = alta carga`:

- **TP**: alta carga predicha y real;
- **TN**: normal predicho y real;
- **FP**: alarma que finalmente no correspondía a alta carga;
- **FN**: no se detectó una situación que realmente era de alta carga.


In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Alta carga"]
))


## 14. El umbral de 0.5 no es una ley

Si queremos evitar especialmente falsos negativos, podemos bajar el umbral.

Eso suele aumentar el recall, a costa de generar más falsos positivos.


In [ ]:
threshold = 0.30
y_pred_030 = (p_high_load >= threshold).astype(int)

print("Umbral:", threshold)
print("Accuracy :", round(accuracy_score(y_test, y_pred_030), 3))
print("Precision:", round(precision_score(y_test, y_pred_030), 3))
print("Recall   :", round(recall_score(y_test, y_pred_030), 3))
print("F1       :", round(f1_score(y_test, y_pred_030), 3))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_030,
    display_labels=["Normal", "Alta carga"]
)
plt.title("Matriz de confusión con umbral 0.30")
plt.show()


## 15. Comparar varios umbrales


In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
rows = []

for threshold in thresholds:
    pred = (p_high_load >= threshold).astype(int)

    rows.append({
        "threshold": round(float(threshold), 1),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })

threshold_metrics = pd.DataFrame(rows)
threshold_metrics


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(threshold_metrics["threshold"], threshold_metrics["precision"], marker="o", label="Precision")
plt.plot(threshold_metrics["threshold"], threshold_metrics["recall"], marker="o", label="Recall")
plt.plot(threshold_metrics["threshold"], threshold_metrics["f1"], marker="o", label="F1")
plt.xlabel("Umbral")
plt.ylabel("Métrica")
plt.title("Efecto del umbral")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 16. Curva ROC y AUC


In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, p_high_load)
auc = roc_auc_score(y_test, p_high_load)

print(f"ROC AUC: {auc:.3f}")


In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Azar")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 17. Probar valores nuevos


In [ ]:
new_windows = pd.DataFrame({
    "flights_15min": [10, 15, 20, 25, 30]
})

new_probabilities = model.predict_proba(new_windows)[:, 1]

pd.DataFrame({
    "flights_15min": new_windows["flights_15min"],
    "p_high_load": new_probabilities
})


## 18. Limitación del modelo

Aquí usamos una sola feature:

```text
flights_15min
```

pero la complejidad ATM real depende de muchas variables.

Con varias features, la parte lineal sería:

$$
z=b+w_1x_1+w_2x_2+...+w_nx_n
$$

y después aplicaríamos exactamente la misma sigmoide.


# 19. Ejercicios

### Ejercicio 1
Calcula manualmente la probabilidad de alta carga para 20 vuelos:

```python
z = intercept + coef * 20
p = 1 / (1 + np.exp(-z))
```

Comprueba que coincide con `predict_proba()`.

### Ejercicio 2
Prueba los umbrales `0.2`, `0.4`, `0.6` y `0.8`.

Observa cómo cambian falsos positivos, falsos negativos, precision y recall.

### Ejercicio 3
Imagina que el modelo alimenta una alerta preventiva ATM.

¿Qué error sería más costoso: un falso positivo o un falso negativo?

### Ejercicio 4
Genera clases muy desbalanceadas y comprueba por qué `accuracy` deja de ser suficiente.

### Ejercicio 5
Añade una segunda feature sintética llamada `conflicts_predicted` y entrena de nuevo el modelo.


# 20. Ideas clave

Al terminar este notebook deberías poder explicar:

1. Por qué la regresión logística sirve para clasificación.
2. Qué hace la función sigmoide.
3. Qué significa `predict_proba()`.
4. La diferencia entre probabilidad y clase.
5. Qué papel tiene el umbral.
6. Qué son TP, TN, FP y FN.
7. La diferencia entre accuracy, precision y recall.
8. Por qué el umbral depende del coste de los errores.
9. Qué representa aproximadamente ROC AUC.
10. Por qué un clasificador no debe evaluarse con una sola métrica.

## Siguiente notebook

`03_overfitting_validation.ipynb`

Estudiaremos qué ocurre cuando un modelo aprende demasiado bien los datos de entrenamiento pero generaliza mal.
